# Model to transform the gas to an hydrogen grid

### Import packages

In [143]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

### Import Data

In [144]:
# Specify the path to your Excel file
input_file_path = '../01_data/01_input_data/02_processed/'
excel_file_path = 'Data_update_V02.xlsx'  

# Read the Excel file into a DataFrame
df_nodes = pd.read_excel(input_file_path + excel_file_path, sheet_name='Nodes')
df_commodities = pd.read_excel(input_file_path + excel_file_path, sheet_name='Commodities')
df_edges = pd.read_excel(input_file_path + excel_file_path, sheet_name='Edges')
df_parameter = pd.read_excel(input_file_path + excel_file_path, sheet_name='Parameters')
df_supply_values = pd.read_excel(input_file_path + excel_file_path, sheet_name='Supply')

### Create input data structure

In [145]:
# Extract nodes, edges and commodities from the DataFrames
Network_nodes = df_nodes['Nodes'].dropna().tolist()
Commodities = df_commodities['Commodities'].dropna().tolist()
Edges = list(zip(df_edges['Source'], df_edges['Destination']))

# Create a nested dictionary for initial capacities
Initial_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    initial_capacity = row['initial_capacities']

    edge = f"{source}{destination}"

    if commodity not in Initial_capacities:
        Initial_capacities[commodity] = {}

    Initial_capacities[commodity][edge] = initial_capacity

# Create a nested dictionary for max capacities
Max_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    max_capacity = row['max_capacities']

    edge = f"{source}{destination}"

    if commodity not in Max_capacities:
        Max_capacities[commodity] = {}

    Max_capacities[commodity][edge] = max_capacity

# Create a nested dictionary for edge cost
Edge_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    edge_cost = row['costs_edge']

    edge = f"{source}{destination}"

    if commodity not in Edge_cost:
        Edge_cost[commodity] = {}

    Edge_cost[commodity][edge] = edge_cost

# Create a nested dictionary for new pipelines
Pipe_new_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    new_cost = row['new_build_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_new_cost:
        Pipe_new_cost[commodity] = {}

    Pipe_new_cost[commodity][edge] = new_cost

# Create a nested dictionary for pipeline conversion
Pipe_conv_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cost = row['conversion_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_cost:
        Pipe_conv_cost[commodity] = {}

    Pipe_conv_cost[commodity][edge] = conv_cost

# Create a nested dictionary for adjusting the capacity when pipeline conversion
Pipe_conv_factor = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cap_factor = row['conversion_capacity_factor']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_factor:
        Pipe_conv_factor[commodity] = {}

    Pipe_conv_factor[commodity][edge] = conv_cap_factor

# Create a nested dictionary for supply values, skipping 0 and NaN values
Supply_values = {}
for index, row in df_supply_values.iterrows():
    commodity = row['Commodity']
    supply_node = row['Node']
    supply_value = row['Supply']

    if commodity not in Supply_values:
        Supply_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(supply_value) and supply_value != 0:
        Supply_values[commodity][supply_node] = supply_value

# Create a nested dictionary for node values, skipping 0 and NaN values
Node_values = {}
for index, row in df_supply_values.iterrows():
    commodity = row['Commodity']
    demand_node = row['Node']
    node_value = row['Supply']

    if commodity not in Node_values:
        Node_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(node_value) and node_value != 0:
        Node_values[commodity][demand_node] = node_value

In [146]:
'''
Create slack nodes for all supply nodes
link a node to supply in case of shortage in the system to all supply nodes
the capacity is infinite but at infinite (super high) cost
'''

# Create a new dictionary for positive values
positive_values_dict = {}

# Iterate through the outer dictionary
for node, values in Node_values.items():
    # Filter out positive values from the inner dictionary
    positive_values = {key: value for key, value in values.items() if value > 0}
    
    # Check if there are positive values before adding to the new dictionary
    if positive_values:
        positive_values_dict[node] = positive_values

print("Dictionary with positive values:", positive_values_dict)

# Get all keys from the inner dictionaries
all_keys = [key for values in positive_values_dict.values() for key in values.keys()]

# Remove duplicates to get unique keys
unique_keys = list(set(all_keys))

print("Unique keys:", unique_keys)

# Create a list with names "shortage_" followed by each key
shortage_list = [f'shortage_{key}' for key in unique_keys]

print("Shortage list:", shortage_list)

#shortage_edges_list = [(key, shortage) for key, shortage in zip(unique_keys, shortage_list)]
shortage_edges_list = [(key, shortage) for key, shortage in zip(shortage_list, unique_keys)]


print("Edges list:", shortage_edges_list)

shoratege_capacity_dict = {commodity: {f'{key}{shortage}': 1000 for key, shortage in zip(shortage_list, unique_keys)} for commodity in Commodities}
shoratege_max_capacity_dict = shoratege_capacity_dict
print("Nested dictionary with capacities:", shoratege_capacity_dict)

shoratege_cost_dict = {commodity: {f'{key}{shortage}': 10000000 for key, shortage in zip(shortage_list, unique_keys)} for commodity in Commodities}
print("Shortage cost:", shoratege_cost_dict)


Dictionary with positive values: {'Methane': {'IN-0003': 1640, 'IN-0004': 2000, 'CS-048A': 2000, 'GS-3435': 2000, 'IN-0007': 2000, 'GS-2348': 2000, 'GS-2350': 2000, 'GS-3152': 2000, 'GS-3219': 2000, 'GS-2347': 2000, 'IN-0013': 2000, 'IN-0014': 2000, 'IN-0015': 2000, 'IN-1336': 2000, 'GS-0870': 2000, 'GS-0877': 2000, 'IN-1339': 2000, 'IN-1340': 2000, 'GS-2280': 2000, 'IN-1342': 2000, 'GS-1634': 2000, 'GS-2026': 2000, 'GS-2005': 2000, 'IN-1346': 2000, 'IN-1347': 2000, 'IN-1348': 2000, 'IN-1349': 2000, 'GS-1997': 2000, 'IN-1351': 2000, 'IN-1352': 2000, 'GS-1423': 2000, 'GS-1424': 2000, 'IN-1355': 2000, 'IN-1356': 2000, 'IN-1357': 2000, 'IN-1358': 2000, 'GS-1658': 2000, 'IN-1360': 2000, 'IN-1361': 2000, 'GS-1293': 2000, 'IN-1363': 2000, 'IN-1364': 2000, 'IN-1365': 2000, 'IN-1366': 2000, 'IN-1367': 2000, 'GS-2277': 2000, 'GS-3455': 2000, 'GS-3454': 2000, 'IN-1371': 2000, 'IN-1372': 2000, 'GS-2203': 2000, 'GS-2202': 2000, 'IN-1375': 2000, 'IN-1376': 2000, 'IN-1377': 2000, 'IN-1378': 2000, 'I

Print data structure for control

In [147]:
# Print the data for control
print("Network Nodes:", Network_nodes)
print("Commodities:", Commodities)
print("Edges:", Edges)
print("Initial Capacities:", Initial_capacities)
print("Max Capacities:", Max_capacities)
print("Costs per edge Capacities:", Edge_cost)
print("Costs for new pipelines:", Pipe_new_cost)
print("Costs for convert pipelines:", Pipe_conv_cost)
print("Conversion capacity factor:", Pipe_conv_factor)
print("Node Values:", Node_values)

Network Nodes: ['IN-0000', 'IN-0001', 'IN-0002', 'IN-0003', 'IN-0004', 'CS-048A', 'GS-3435', 'IN-0007', 'GS-2348', 'GS-2350', 'GS-3152', 'GS-3219', 'GS-2347', 'IN-0013', 'IN-0014', 'IN-0015', 'IN-0016', 'IN-0017', 'IN-0018', 'IN-0019', 'IN-0021', 'IN-0022', 'GS-2465', 'IN-0024', 'IN-0025', 'IN-0026', 'GS-2544', 'GS-3413', 'GS-2514', 'IN-0030', 'IN-0031', 'GS-3416', 'IN-0033', 'IN-0034', 'IN-0035', 'GS-3473', 'GS-0777', 'GS-2074', 'GS-1837', 'GS-2533', 'IN-0041', 'IN-0042', 'IN-0043', 'IN-0044', 'IN-0045', 'IN-0046', 'IN-0047', 'IN-0048', 'GS-4407', 'IN-0050', 'IN-0051', 'IN-0052', 'GS-2355', 'IN-0054', 'GS-7249', 'GS-7231', 'GS-7232', 'IN-0058', 'IN-0059', 'IN-0060', 'IN-0061', 'IN-0062', 'IN-0063', 'GS-6443', 'GS-6607', 'GS-6606', 'IN-0067', 'IN-0068', 'IN-0069', 'IN-0070', 'GS-4707', 'IN-0072', 'IN-0073', 'IN-0074', 'IN-0075', 'GS-6927', 'IN-0077', 'GS-6513', 'GS-6668', 'IN-0080', 'GS-3230', 'IN-0082', 'GS-6821', 'IN-0084', 'GS-6612', 'IN-0086', 'IN-0087', 'GS-4603', 'GS-6560', 'IN-0

In [148]:
#implement factor to adjust capacity when conversion from methane to hydrogen
#TODO Implement it from the input file and use a correct factor
conversion_factor = Pipe_conv_factor

## Model

### Create model

In [149]:
# Create a new model
model = gp.Model("Grid_Transformation")

### Define parameters

In [150]:
# Parameters
commodities = Commodities  # Commodity types
#real network elements
network_nodes = Network_nodes # Nodes of the system
network_edges = Edges  # Edges
initial_capacities = Initial_capacities # Initial capacities
max_capacities = Max_capacities  # Maximum capacities
costs_edge = Edge_cost  # Cost to transport from node to node
capacity_new_cost = Pipe_new_cost  # Cost to increase capacity
capacity_change_cost = Pipe_conv_cost  # Cost to increase capacity
node_value = Node_values #contains supply and demand values

#slack parameters
slack_nodes = shortage_list
slack_edges = shortage_edges_list
slack_capacities = shoratege_capacity_dict
slack_cost = shoratege_cost_dict

#complete network of the model
all_edges = network_edges + shortage_edges_list

### Define decision variables

In [151]:
# Decision variables
x_flow = {} #flow of commodity on an edge
x_flow_shortage = {}
y_new_cap = {} #new build capacity for a commodity on an edge between two edges
z_conv_cap = {} #capacity of a commodity converted on an edge between two nodes
Change = {} # Binary variable for switching

for commodity in commodities:
    x_flow[commodity] = {}
    x_flow_shortage[commodity] = {}
    y_new_cap[commodity] = {}
    z_conv_cap[commodity] = {}
    Change[commodity] = {}
    for edge in all_edges:
        x_flow[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_{commodity}_{edge[0]}_{edge[1]}")
        y_new_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"y_new_cap{commodity}_{edge[0]}_{edge[1]}")
        z_conv_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"z_conv_cap{commodity}_{edge[0]}_{edge[1]}")
        Change[commodity][edge] = model.addVar(vtype=GRB.BINARY, name=f"change_{commodity}_{edge[0]}_{edge[1]}")
    for edge in slack_edges:
        x_flow_shortage[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_shortage_{commodity}_{edge[0]}_{edge[1]}")
model.update()

### Define objective and constraints

In [152]:
# Objective function (minimize total transportation cost + cost to increase and convert capacity)
model.setObjective(
    gp.quicksum(x_flow[commodity][edge] * costs_edge[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges) +
    gp.quicksum(y_new_cap[commodity][edge] * capacity_new_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges) +
    gp.quicksum(z_conv_cap[commodity][edge] * capacity_change_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges),
    #gp.quicksum(x_flow_shortage[commodity][edge] * slack_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in slack_edges),
    GRB.MINIMIZE
)

# Constraints

#inflow of a node must equal the outflow of a node
for node in network_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    - gp.quicksum(x_flow[commodity][edge] for edge in network_edges if edge[0] == node)
                                    + node_value[commodity][node] 
                                    == 0, f"flow_constraint_{commodity}_{node}")

#slack node at the sources to avoid infeasible problems.        

# Supply constraints for the second commodity
for node in slack_nodes:
    for commodity in commodities: 
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[0] == node) >= 0, f"supply_shortage_{commodity}_{node}")


# Add constraint for equality between x_flow and x_flow_shortage for the same edge
for commodity in commodities:
    for edge in slack_edges:
        if (edge[0], edge[1]) in slack_edges or (edge[1], edge[0]) in slack_edges:
            # Ensure equality for the corresponding edges
            model.addConstr(x_flow[commodity][edge] == x_flow_shortage[commodity][(edge[0], edge[1])], 
                            f"equality_constraint_{commodity}_{edge}")


#Capacity constraint for flow
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(x_flow[commodity][edge] 
                        <= y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] #+ initial_capacities[commodity][f"{edge[0]}{edge[1]}"]
                        , f"used_capacity_{commodity}_{edge[0]}_{edge[1]}")

# Capacity constraint for maximal capacity
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] 
                        <= max_capacities[commodity][f"{edge[0]}{edge[1]}"], f"capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraints for edge conversion
for edge in network_edges:
    model.addConstr(Change[Commodities[0]][edge] + Change[Commodities[1]][edge] == 1, f"switching_constraint_{edge[0]}_{edge[1]}")

for commodity in commodities:
    for edge in network_edges:
        model.addConstr(initial_capacities[Commodities[0]][f"{edge[0]}{edge[1]}"] * Change[commodity][edge] #* conversion_factor[commodity][node]
                        == z_conv_cap[commodity][edge], f"changed_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraing no negative flow
for commodity in commodities:
    for edge in all_edges:
        model.addConstr(x_flow[commodity][edge] >= 0, f"non_negativity_x_{commodity}_{edge[0]}_{edge[1]}")

### Optimize the model

In [153]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 93008 rows, 70100 columns and 190220 nonzeros
Model fingerprint: 0xdec400d9
Variable types: 52648 continuous, 17452 integer (17452 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+04]
  Objective range  [1e+00, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+07]
Presolve removed 18036 rows and 1168 columns
Presolve time: 0.03s

Explored 0 nodes (0 simplex iterations) in 0.15 seconds (0.04 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -


### Results processing

In [154]:
# Print the results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found!")
    for commodity in commodities:
        for edge in network_edges:
            print(f"{commodity}, {edge}: "
                  f"Flow of Commodity = {x_flow[commodity][edge].x}, "
                  f"New Capacity = {y_new_cap[commodity][edge].x}, "
                  f"Switched = {Change[commodity][edge].x}, "
                  f"Changed Capacity = {z_conv_cap[commodity][edge].x}, ")
        for edge in slack_edges:    
           print(f"{commodity}, {edge}: "
                 f"Flow of Shortage = {x_flow_shortage[commodity][edge].x}")
    print("****************************")
    print(f"Total cost: {model.objVal}")
else:
    print("No optimal solution found.")

No optimal solution found.


In [155]:
# Assuming commodities, edges, x_flow, y_new_cap, Change, and z_conv_cap are defined in your code

# Create lists to store the data
results_data = []
columns = ["Commodity", "Edge", "Flow", "New Capacity", "Switched", "Changed Capacity"]
shortage_data = []
shortage_columns = ["Commodity", "Edge", "Flow"]

# Check if the model has an optimal solution
if model.status == GRB.OPTIMAL:
    for commodity in commodities:
        for edge in network_edges:
            # Append data to the list
            results_data.append([commodity,
                                 edge, 
                                 x_flow[commodity][edge].x, 
                                 y_new_cap[commodity][edge].x, 
                                 Change[commodity][edge].x, 
                                 z_conv_cap[commodity][edge].x])
        for edge in slack_edges:    
            shortage_data.append([commodity, 
                              edge, 
                              x_flow_shortage[commodity][edge].x])
else:
    print("No optimal solution found.")

if model.status == GRB.OPTIMAL:
    # Create a DataFrame
    results_df = pd.DataFrame(results_data, columns=columns)
    shortage_df = pd.DataFrame(shortage_data, columns=shortage_columns)
    # Print the DataFrame
    print(results_df)
    print('********************************')
    print(shortage_df)

No optimal solution found.


In [156]:
#model.write()